# 🚀 Aplicación Web AEMET (FastAPI + Streamlit)
¡Hola! En este notebook vamos a integrar el backend (FastAPI) y el frontend (Streamlit) 
que Tamara intentó hacer. ¡Lo dejaremos 100% funcional!

**¿Qué hace esto?**
1. Crea los archivos de la API y de Streamlit.
2. Inicia los servidores en segundo plano.
3. Se queda escuchando indefinidamente para que puedas acceder a la web.



In [ ]:
# Instalamos las dependencias necesarias en el entorno actual de Jupyter
import sys
!{sys.executable} -m pip install -q streamlit fastapi uvicorn psycopg[binary] requests pandas pydantic


In [ ]:
%%writefile api_aemet.py
# -*- coding: utf-8 -*-
"""
API backend con FastAPI para predicciones y consulta histórica.
"""

from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel
import psycopg
from datetime import date

app = FastAPI(title="AEMET", description="API de AEMET", version="1.0")

# ---- Credenciales de BD ----
HOST = "database-dai07rt-proyecto-aemet-2026.cr828ecqk1a6.eu-central-1.rds.amazonaws.com"
PORT = "5432"
DBNAME = "postgres"
USER = "aemet2026"
PASSWORD = "mondongo-dai07rt-aemet-2026"
DSN = f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}"

class InputFeatures(BaseModel):
    features: list

@app.get("/")
def bienvenida():
    return {"mensaje": "Bienvenido a la API de AEMET. Ve a /docs para más info."}

@app.post("/modelos_max/{idema}/predict")
def prediccion_temp_max(idema: str, input_data: InputFeatures):
    """Simula una predicción de temperatura máxima."""
    print(f"Calculando temp máxima para {idema}...")
    valor_simulado = input_data.features[0] * 1.5 + 5.0
    return {
        "status": "success",
        "idema": idema,
        "mensaje": "Predicción de temperatura máxima calculada",
        "prediccion": [[valor_simulado]]
    }

@app.post("/modelos_min/{idema}/predict")
def prediccion_temp_min(idema: str, input_data: InputFeatures):
    """Simula una predicción de temperatura mínima."""
    print(f"Calculando temp mínima para {idema}...")
    valor_simulado = input_data.features[0] * 0.8 - 2.0
    return {
        "status": "success",
        "idema": idema,
        "mensaje": "Predicción de temperatura mínima calculada",
        "prediccion": [[valor_simulado]]
    }

@app.get("/historico/obtener_historico")
def obtener_historico(
    idema: str = Query(..., alias="id"),
    fecha_inicio: date = Query(...),
    fecha_fin: date = Query(...)
):
    """Consulta el histórico directamente a la base de datos."""
    if fecha_inicio > fecha_fin:
        raise HTTPException(status_code=400, detail="Fechas incorrectas")
        
    print(f"Consultando histórico de {idema}...")
    try:
        with psycopg.connect(DSN) as conn:
            with conn.cursor() as cur:
                query = """
                    SELECT fecha, tmax, tmed, tmin 
                    FROM datos_climaticos 
                    WHERE indicativo = %s AND fecha >= %s AND fecha <= %s
                    ORDER BY fecha;
                """
                cur.execute(query, (idema, fecha_inicio, fecha_fin))
                filas = cur.fetchall()
                
        registros = []
        for fila in filas:
            registros.append({
                "fecha": str(fila[0]),
                "tmax": float(fila[1]) if fila[1] else None,
                "tmed": float(fila[2]) if fila[2] else None,
                "tmin": float(fila[3]) if fila[3] else None
            })
            
        return {"registros": registros}
    except Exception as e:
        print(f"Error en BD: {e}")
        raise HTTPException(status_code=500, detail="Error al conectar a la BD")




In [ ]:
%%writefile web_aemet.py
# -*- coding: utf-8 -*-
"""
Frontend con Streamlit.
"""

import streamlit as st
import requests
import pandas as pd
import plotly.express as px
from datetime import date

st.set_page_config(layout="wide")
FASTAPI_URL = "http://127.0.0.1:8000"

st.title("☀️ Panel de Control - AEMET")
st.text("Exploración de datos históricos y predicciones.")

with st.sidebar:
    st.header("Configuración")
    opcion = st.selectbox(
        "¿Qué quieres hacer?",
        ["Temperatura Máxima", "Temperatura Mínima", "Histórico"]
    )
    
if opcion in ["Temperatura Máxima", "Temperatura Mínima"]:
    st.header(f"Predicción de {opcion}")
    idema = st.text_input("Código de la estación (IDEMA):", value="0009X")
    valor = st.number_input("Valor numérico de entrada (ej. temperatura actual):", value=15.0)
    
    if st.button("Calcular"):
        with st.spinner("Conectando con la API..."):
            ruta = "modelos_max" if opcion == "Temperatura Máxima" else "modelos_min"
            url = f"{FASTAPI_URL}/{ruta}/{idema}/predict"
            res = requests.post(url, json={"features": [valor]})
            
            if res.status_code == 200:
                data = res.json()
                st.success(data["mensaje"])
                
                # Accedemos a la predicción simulada
                valor_pred = data["prediccion"][0][0]
                st.metric(label=f"Predicción ({idema})", value=f"{valor_pred:.2f} °C")
            else:
                st.error("Hubo un error al conectar con la API.")

elif opcion == "Histórico":
    st.header("Histórico de Temperaturas")
    idema = st.text_input("Código IDEMA:", value="3195")
    col1, col2 = st.columns(2)
    with col1:
        f_ini = st.date_input("Fecha Inicio", value=date(2020, 1, 1))
    with col2:
        f_fin = st.date_input("Fecha Fin", value=date(2020, 1, 31))
        
    if st.button("Buscar"):
        with st.spinner("Buscando en la base de datos..."):
            url = f"{FASTAPI_URL}/historico/obtener_historico"
            res = requests.get(url, params={"id": idema, "fecha_inicio": f_ini, "fecha_fin": f_fin})
            
            if res.status_code == 200:
                data = res.json()
                registros = data.get("registros", [])
                if registros:
                    df = pd.DataFrame(registros)
                    df["fecha"] = pd.to_datetime(df["fecha"])
                    fig = px.line(df, x="fecha", y=["tmax", "tmed", "tmin"], title="Evolución de Temperaturas", markers=True)
                    st.plotly_chart(fig, use_container_width=True)
                    st.dataframe(df)
                else:
                    st.warning("No se encontraron datos para esas fechas.")
            else:
                st.error("Error al consultar el histórico.")




## 🚀 Ejecución de los servidores
En esta celda arrancaremos ambos servicios (FastAPI y Streamlit) en segundo plano usando `subprocess`.


In [ ]:
import subprocess
import time
import sys

print("Iniciando la API (FastAPI)...")
proceso_api = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api_aemet:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

print("Iniciando la Web (Streamlit)...")
proceso_web = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "web_aemet.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("¡Todo listo! Accede a http://localhost:8501 en tu navegador.")



## 🔄 Mantener activo
Dejaremos este bucle infinito para que los procesos no mueran y puedas interactuar con la aplicación tranquilamente.
Para detenerlo, simplemente interrumpe la celda.


In [ ]:
try:
    print("Servidores activos. Presiona Stop en Jupyter para detener.")
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    print("Deteniendo servidores...")
    proceso_api.terminate()
    proceso_web.terminate()
    print("¡Servidores detenidos!")

